# Convert Tubes To Images

This notebook contains a few examples of how to call wrapped methods in itk and ITKTubeTK.

ITK, ITKTubeTK, and ITKWidgets must be installed on your system for this notebook to work.

In [11]:
import os
import sys
import numpy

In [4]:
import itk
from itk import TubeTK as ttk
# from itkwidgets import view

Load the tubes and a reference image which provides the size, spacing, origin, and orientation for the desired output image.

In [6]:
PixelType = itk.F
Dimension = 3
ImageType = itk.Image[PixelType, Dimension]
    
# Read tre file
TubeFileReaderType = itk.SpatialObjectReader[Dimension]
    
tubeFileReader = TubeFileReaderType.New()
tubeFileReader.SetFileName("/home/hsalles/Téléchargements/tubetk/Normal-002/AuxillaryData/VascularNetwork.tre")
tubeFileReader.Update()

tubes = tubeFileReader.GetGroup()


# Read template image
TemplateImageType = itk.Image[PixelType, Dimension]
TemplateImageReaderType = itk.ImageFileReader[TemplateImageType]
    
templateImageReader = TemplateImageReaderType.New()
templateImageReader.SetFileName("/home/hsalles/Téléchargements/tubetk/Normal-002/MRA/Normal002-MRA.mha")
templateImageReader.Update()

templateImage = templateImageReader.GetOutput()

Visualize the template image, just because it looks cool - the data in the image is actually irrelevant.

In [ ]:
# view(templateImage)

Viewer(geometries=[], gradient_opacity=0.22, point_sets=[], rendered_image=<itk.itkImagePython.itkImageF3; pro…

Create a binary image that represents the spatial extent of the TubeSpatialObjects in the hierarchy of SpatialObjects in the variable "tubes" that was read-in above.   If you only want to visualize centerlines of the tubes, set "UseRadius" to false.

In [ ]:
TubesToImageFilterType = ttk.ConvertTubesToImage[TemplateImageType]
tubesToImageFilter = TubesToImageFilterType.New()
tubesToImageFilter.SetUseRadius(True)
tubesToImageFilter.SetTemplateImage(templateImageReader.GetOutput())
tubesToImageFilter.SetInput(tubes)
tubesToImageFilter.Update()

outputImage: itk.itkImagePython.itkImageF3 = tubesToImageFilter.GetOutput()

In [10]:
# convert outputImage to mha file
OutputImageWriterType = itk.ImageFileWriter[TemplateImageType]
outputImageWriter = OutputImageWriterType.New()
outputImageWriter.SetFileName("/home/hsalles/Téléchargements/tubetk/Normal-002/AuxillaryData/VascularNetwork.mha")
outputImageWriter.SetInput(outputImage)
outputImageWriter.Update()

Visualize the results by blending the template and output images.  Again, the content of the template image
doesn't actually matter, but since these tubes were generated from the content of the template image, blending them illustrates how well the binary tube image corresponds with their source image.

In [8]:
TTKImageMathType = ttk.ImageMath[ImageType,ImageType]

imMath = TTKImageMathType.New(Input = outputImage)
imMath.AddImages(templateImage, 2048, 1)
combinedImage = imMath.GetOutput()
# view(combinedImage)

TemplateTypeError: itk.ImageMath is not wrapped for input type `itk.Image[itk.F,3], itk.Image[itk.F,3]`.

To limit the size of the package, only a limited number of
types are available in ITK Python. To print the supported
types, run the following command in your python environment:

    itk.ImageMath.GetTypes()

Possible solutions:
* If you are an application user:
** Convert your input image into a supported format (see below).
** Contact developer to report the issue.
* If you are an application developer, force input images to be
loaded in a supported pixel type.

    e.g.: instance = itk.ImageMath[itk.Image[itk.SS,2]].New(my_input)

* (Advanced) If you are an application developer, build ITK Python yourself and
turned to `ON` the corresponding CMake option to wrap the pixel type or image
dimension you need. When configuring ITK with CMake, you can set
`ITK_WRAP_${type}` (replace ${type} with appropriate pixel type such as
`double`). If you need to support images with 4 or 5 dimens